# Recommendations demo

**Goal:** for a given input item, retrieve and compare the recommendations
produced by all three models in this project -- TTN ("Complete the Look",
complementary products), SigLIP2 ("Visually Similar Products",
substitutes), and Popularity ("Trending in Category") -- using each
model's current **champion**, as recorded in that model's own
`champion_selection_*.json`, not an arbitrary version chosen ad hoc here.

This is a different kind of model access than the live API
(`api.py`)/demo page (`demo/index.html`) uses: there, every request is a
plain SQLite lookup against work already done offline. Here:

- **TTN's and Popularity's champions already have precomputed
  recommendation tables on disk** (`complements.parquet` /
  `popularity.parquet`, written by each model's own
  `generate_recommendations.py` / `build_popularity.py`) -- for those two,
  this notebook just reads the existing table, the same as the live API
  would, no model execution involved.
- **SigLIP2's champion (the imported `pilot_200k` adapter) has no
  precomputed recommendation table yet** -- nothing in this repo has run
  it across the catalogue and saved the result. So for SigLIP2 specifically,
  this notebook loads the live adapter checkpoint, as a first step toward
  running it over the full catalogue and storing those results, the same
  way the other two models already work.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# demo/recommendations_demo.ipynb's kernel cwd is this notebook's own folder
# (demo/) when opened normally -- walk up to the repo root so the SigLIP2
# package import below and every path resolved later are correct regardless
# of how the kernel was launched.
ROOT = Path.cwd()
if not (ROOT / "SigLIP2").exists():
    ROOT = ROOT.parent
assert (ROOT / "SigLIP2").exists(), f"couldn't find the repo root from {Path.cwd()}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from SigLIP2.adapter import load_adapter

print(f"repo root: {ROOT}")

repo root: /Users/lazr/PycharmProjects/RecSystem


In [2]:
# Read each model's own champion-selection record. Each file is hand-edited,
# not written by any script -- see this session's discussion for why the
# three schemas differ: TTN's candidates are always {snapshot_id,
# version_id} (version numbering restarts per snapshot, so both fields are
# required to identify one model); SigLIP2's aren't uniformly snapshot-scoped
# (the imported pilot_200k checkpoint isn't tied to any --window-days
# snapshot), so it uses a direct checkpoint_path instead; Popularity has no
# training/versioning at all (build_popularity.py is deterministic), so its
# only field is which snapshot's output this is.
CHAMPIONS = {
    "ttn": json.loads((ROOT / "TTN" / "champion_selection_ttn.json").read_text()),
    "siglip2": json.loads((ROOT / "SigLIP2" / "champion_selection_siglip.json").read_text()),
    "popularity": json.loads((ROOT / "Popularity" / "champion_selection_popularity.json").read_text()),
}
for name, c in CHAMPIONS.items():
    print(name, {k: v for k, v in c.items() if k != "_note"})

# --- TTN: champion already has precomputed recommendations -- just resolve
# the path, no model loading needed.
ttn_snapshot = CHAMPIONS["ttn"]["snapshot_id"]
ttn_version = CHAMPIONS["ttn"]["version_id"]
ttn_complements_path = (ROOT / "TTN" / "generated" / ttn_snapshot
                         / "recommendations" / "complements.parquet")
assert ttn_complements_path.exists(), f"missing: {ttn_complements_path}"
print(f"\nTTN champion recommendations -> {ttn_complements_path.relative_to(ROOT)}")

# --- Popularity: champion is just a snapshot pointer -- resolve the path,
# no model/version concept at all.
popularity_snapshot = CHAMPIONS["popularity"]["snapshot_id"]
popularity_path = (ROOT / "Popularity" / "generated" / popularity_snapshot
                    / "recommendations" / "popularity.parquet")
assert popularity_path.exists(), f"missing: {popularity_path}"
print(f"Popularity champion recommendations -> {popularity_path.relative_to(ROOT)}")

# --- SigLIP2: champion has no precomputed recommendations yet -- load the
# live adapter checkpoint instead.
siglip_checkpoint_path = ROOT / CHAMPIONS["siglip2"]["checkpoint_path"]
assert siglip_checkpoint_path.exists(), f"missing: {siglip_checkpoint_path}"
siglip_adapter = load_adapter(checkpoint_path=siglip_checkpoint_path)
print(f"SigLIP2 champion adapter loaded ({type(siglip_adapter).__name__}) "
      f"from {siglip_checkpoint_path.relative_to(ROOT)}")

ttn {'snapshot_id': 'w60_2017-12-09', 'version_id': '2026-10-01_v_001'}
siglip2 {'checkpoint_path': 'SigLIP2/checkpoints/pilot_200k/taxonomy_w20_random_200k.pt'}
popularity {'snapshot_id': 'w60_2017-12-09'}

TTN champion recommendations -> TTN/generated/w60_2017-12-09/recommendations/complements.parquet
Popularity champion recommendations -> Popularity/generated/w60_2017-12-09/recommendations/popularity.parquet
SigLIP2 champion adapter loaded (ContrastiveAdapters) from SigLIP2/checkpoints/pilot_200k/taxonomy_w20_random_200k.pt


## Compare TTN's two recommendation approaches

Both files below come from the exact same trained champion model
(`ttn_snapshot`/`ttn_version` above) -- the only thing that differs is the
GENERATION-time merge rule across a query's multiple licensed target
categories:

- **`complements.parquet`** (the "20 most relevant items") -- pools every
  licensed category's candidates into one flat pool and keeps the single
  overall top-20 by score. Can let one category dominate entirely if it
  happens to score higher -- verified this session on a real item: its
  flat top-20 was 20/20 from a single category, ignoring 4 other licensed
  categories with real co-purchase evidence behind them.
- **`complements_proportional.parquet`** (the "weighted 20 items") --
  splits the 20 slots across licensed categories in proportion to each
  category's co-purchase evidence (`edges`, from
  `complementary_categories.pkl`), then ranks independently *within* each
  category -- guarantees every licensed category with a nonzero share
  gets represented, at the cost of sometimes filling a slot with a weaker
  same-category match instead of a stronger one from an under-allocated
  category.

In [ ]:
# Resolve the proportional file's path too -- same champion snapshot/version
# as ttn_complements_path above, just a different generation script's output
# (TTN/generate_recommendations_proportional.py instead of
# TTN/generate_recommendations.py).
ttn_proportional_path = (ROOT / "TTN" / "generated" / ttn_snapshot
                          / "recommendations" / "complements_proportional.parquet")
assert ttn_proportional_path.exists(), f"missing: {ttn_proportional_path}"
print(f"TTN proportional recommendations -> {ttn_proportional_path.relative_to(ROOT)}")

ttn_flat = pd.read_parquet(ttn_complements_path)
ttn_proportional = pd.read_parquet(ttn_proportional_path)


def compare_ttn(query_asin: str):
    """Side-by-side look at both TTN recommendation approaches for one item."""
    flat_rows = ttn_flat[ttn_flat["query_asin"] == query_asin]
    prop_rows = ttn_proportional[ttn_proportional["query_asin"] == query_asin]
    if flat_rows.empty and prop_rows.empty:
        print(f"{query_asin} has no TTN recommendations in either file.")
        return

    print(f"query_asin: {query_asin}")
    print(f"\n--- complements.parquet (20 most relevant, flat merge) ---")
    print(flat_rows[["rank", "candidate_asin", "score"]].to_string(index=False))

    print(f"\n--- complements_proportional.parquet (weighted 20, by category evidence) ---")
    print(prop_rows[["rank", "candidate_asin", "score", "target_node_id"]].to_string(index=False))

    print(f"\ndistinct categories -- flat: n/a (not tracked in this file), "
          f"proportional: {prop_rows['target_node_id'].nunique()}")


# Pick a sample item that actually has recommendations in both files -- the
# same one used earlier this session to demonstrate the flat merge's
# single-category-domination problem.
SAMPLE_ASIN = "0560467893"
compare_ttn(SAMPLE_ASIN)